# grad-tracking-global-toggle — ex2: no_grad decorator built on the module-level toggle (try/finally restore)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `grad-tracking-global-toggle`. Running the final beacon cell reports progress against the `Backprop: Grad-tracking toggle` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Grad-tracking toggle` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grad-tracking-global-toggle`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grad-tracking-global-toggle"
DD_SUBTOPIC = "Backprop: Grad-tracking toggle"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Grad-tracking global toggle — quick refresher

A module-level `grad_tracking_enabled` boolean gates all autograd. The **decorator** form wraps a whole function in an automatic disable/restore.

**Worked exemplar.**
```python
@no_grad
def update_ema(target, source, decay=0.99):
    # inside this body, grad_tracking_enabled is False; restored on return
    ...
```
Even if the body raises, the previous value must be restored — `try/finally`.

### Exercise 2 — no_grad decorator built on the module-level toggle (try/finally restore)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply a decorator pattern over the module-level grad_tracking_enabled toggle: disable on entry, restore previous value on exit even when the decorated function raises.
> Keywords: no-grad, decorator, try-finally, exception-safety
> ```

**KCs targeted:** `grad-tracking-global-toggle`, `no-grad-decorator-restores-on-exception`

Implement `no_grad` as a DECORATOR (not a context manager).

1. `no_grad(fn)` returns a wrapper that:
   - Snapshots the current value of the global `grad_tracking_enabled`.
   - Sets `grad_tracking_enabled = False` before calling `fn`.
   - Uses `try / finally` so the previous value is restored even if `fn` raises.
   - Returns whatever `fn` returns.

**Critical: assign to the GLOBAL.** Inside the wrapper, you MUST do `global grad_tracking_enabled` before assigning. Without it, the wrapper creates a local variable and the global never flips — silent no-op bug.

The test verifies: (a) inside the decorated fn the toggle is False; (b) after return the toggle is restored to its prior value; (c) if the decorated fn raises, the toggle is STILL restored; (d) nesting works (decorate inside `with NoGrad(): ...` keeps the toggle False both during and after); (e) the wrapper preserves the return value.

In [ ]:
def no_grad(fn):
    def wrapper(*args, **kwargs):
        global grad_tracking_enabled
        prev = grad_tracking_enabled
        grad_tracking_enabled = False
        try:
            return fn(*args, **kwargs)
        finally:
            grad_tracking_enabled = prev
    return wrapper


<details><summary>Solution</summary>

```python
def no_grad(fn):
    def wrapper(*args, **kwargs):
        global grad_tracking_enabled
        prev = grad_tracking_enabled
        grad_tracking_enabled = False
        try:
            return fn(*args, **kwargs)
        finally:
            grad_tracking_enabled = prev
    return wrapper
```

**Why `try / finally` and not `try / except`.** We don't want to swallow the exception — propagate it. We only want to guarantee the restore. `finally` runs whether the body returned, raised, or even called `sys.exit`.

**Why snapshot `prev` instead of `True`.** If `no_grad` is called while the toggle is ALREADY False (nested call, or after an outer `with NoGrad():`), unconditionally restoring to True would re-enable grad tracking inside an outer no-grad scope — bug. Saving and restoring the previous value composes correctly.

**The `global` keyword.** Without it, `grad_tracking_enabled = False` creates a function-local variable and the module-level toggle never changes. This is the most common bug when implementing toggle decorators in Python — the test specifically checks the global flips.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()